# Colab Heavy Models - MedGemma + ColPali

**Run this on Google Colab or Kaggle with a GPU (free T4, 16 GB).**

The two mandatory heavy models do not fit on a 4 GB local GPU, so this notebook
runs them in the cloud and produces artifacts the rest of the pipeline consumes:

| Output | Produced by |
|---|---|
| `results/generated_reports.csv` | MedGemma - Report Generation Mode |
| `results/qa_answers.csv` | MedGemma - QA Mode (with / without RAG context) |
| `results/colpali_index/` | ColPali - retrieval index over the corpus |
| `results/colpali_finetuned/` | ColPali LoRA fine-tune (stretch) |

**Before running:** `Runtime -> Change runtime type -> GPU`.


## 0. GPU check


In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 1. Install dependencies


In [ ]:
!pip install -q "transformers>=4.40.0" "colpali-engine>=0.3.0,<0.4.0" accelerate bitsandbytes
!pip install -q faiss-cpu pandas pillow tqdm peft
print("Dependencies installed.")


## 2. Get the repo + data onto Colab

**Recommended: upload the repo as a zip.** Zip your local `DSAI-413-Assignment2`
folder (include `data/processed/` and `data/qa/` which you built locally) and
upload it with the cell below.

Alternatively clone from GitHub (if pushed) or mount Google Drive.


In [ ]:
# Option A: upload a zip of the repo (recommended)
import zipfile

try:
    from google.colab import files
    print("Select your DSAI-413-Assignment2 zip ...")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".zip"):
            with zipfile.ZipFile(fname) as z:
                z.extractall("/content")
            print("Extracted", fname)
except Exception as e:
    print("Skipping upload (not on Colab or cancelled):", e)

# Option B: clone (requires the repo pushed to GitHub)
# !git clone https://github.com/Ziadshaaban/DSAI-413-Assignment2.git /content/DSAI-413-Assignment2

# Option C: mount Drive
# from google.colab import drive; drive.mount('/content/drive')


In [ ]:
import sys, os
from pathlib import Path

# Auto-locate the repo root (edit if it lives elsewhere)
candidates = [
    Path("/content/DSAI-413-Assignment2"),
    Path("/kaggle/working/DSAI-413-Assignment2"),
    Path.cwd(),
]
PROJECT_ROOT = next((c for c in candidates if (c / "config.py").exists()), Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

import config  # noqa
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


## 2b. Remap image paths to Kaggle filesystem

In [ ]:
import re

def _remap_image_paths(df: pd.DataFrame) -> pd.DataFrame:
    """Replace Windows-absolute image_path prefix with the correct Kaggle/Colab path."""
    # Kaggle mounts added datasets at /kaggle/input/<dataset-slug>/
    candidates = [
        Path("/kaggle/input/mimic-cxr-dataset/official_data_iccv_final"),
        Path("/kaggle/input/mimic-cxr-dataset"),
        Path("/content/mimic-cxr-dataset/official_data_iccv_final"),
    ]
    image_root = next((c for c in candidates if (c / "files").exists()), None)

    if image_root is None:
        print("WARNING: could not auto-detect image root; image_path left unchanged.")
        return df

    print(f"Image root: {image_root}")

    def _remap(p):
        if pd.isna(p) or not str(p).strip():
            return p
        m = re.search(r'(files[\\/].+)', str(p))
        if m:
            return str(image_root / m.group(1).replace("\\", "/"))
        return p

    df = df.copy()
    df["image_path"] = df["image_path"].apply(_remap)
    n_ok = df["image_path"].apply(lambda p: Path(str(p)).exists()).sum()
    print(f"{n_ok}/{len(df)} image paths verified on disk.")
    return df


corpus_df = _remap_image_paths(pd.read_csv(config.PROCESSED_DATA_DIR / "corpus.csv"))
eval_df   = _remap_image_paths(pd.read_csv(config.PROCESSED_DATA_DIR / "eval.csv"))
print("corpus:", len(corpus_df), "| eval:", len(eval_df))

## 3. MedGemma - load model

MedGemma 4B is the heaviest component. On a T4, load in bfloat16; if you hit OOM
during batch generation, set `USE_4BIT = True`.


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MEDGEMMA_ID = "google/medgemma-4b-it"
USE_4BIT = False  # set True if you hit OOM

if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    medgemma = AutoModelForImageTextToText.from_pretrained(
        MEDGEMMA_ID, quantization_config=bnb, device_map="auto")
else:
    medgemma = AutoModelForImageTextToText.from_pretrained(
        MEDGEMMA_ID, torch_dtype=torch.bfloat16, device_map="auto")

mg_processor = AutoProcessor.from_pretrained(MEDGEMMA_ID)
medgemma.eval()
print("MedGemma loaded.")


### MedGemma generation helpers

Inline here so the notebook is self-contained. The same logic lives in
`src/models/medgemma.py` for local / API use.


In [ ]:
RADIOLOGIST_SYSTEM = ("You are an expert radiologist. Write concise, clinically "
                      "accurate chest X-ray reports.")


def medgemma_generate(image, prompt, system=RADIOLOGIST_SYSTEM, max_new_tokens=300):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system}]},
        {"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image", "image": image},
        ]},
    ]
    inputs = mg_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(medgemma.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        gen = medgemma.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return mg_processor.decode(gen[0][input_len:], skip_special_tokens=True).strip()


def medgemma_report(image):
    prompt = "Provide a structured chest X-ray report with FINDINGS and IMPRESSION sections."
    return medgemma_generate(image, prompt, max_new_tokens=300)


def medgemma_answer(image, question, context=None):
    if context:
        prompt = ("Use the retrieved similar reports as reference context, but base "
                  "your answer primarily on the X-ray image.\n\n"
                  f"Reference context:\n{context}\n\n"
                  f"Question: {question}\nAnswer concisely.")
    else:
        prompt = f"Question about this chest X-ray: {question}\nAnswer concisely."
    return medgemma_generate(image, prompt, max_new_tokens=150)


### Quick test on one image


In [ ]:
eval_csv = config.PROCESSED_DATA_DIR / "eval.csv"
assert eval_csv.exists(), f"Missing {eval_csv} - run src/data/preprocess.py locally first."
# eval_df already loaded and remapped in cell 2b
print("Eval set:", len(eval_df), "studies")

sample = eval_df.iloc[0]
sample_img = Image.open(sample["image_path"]).convert("RGB")
print("GROUND TRUTH REPORT:\n", sample["text"][:300])
print("\nMEDGEMMA REPORT:\n", medgemma_report(sample_img))

### 3a. Batch report generation -> `results/generated_reports.csv`


In [ ]:
records = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="MedGemma reports"):
    try:
        img = Image.open(row["image_path"]).convert("RGB")
        gen = medgemma_report(img)
    except Exception as e:
        gen = f"[ERROR] {e}"
    records.append({
        "study_id": row["study_id"],
        "image_path": row["image_path"],
        "ground_truth": row["text"],
        "medgemma_direct": gen,
    })

reports_df = pd.DataFrame(records)
out_path = config.RESULTS_DIR / "generated_reports.csv"
reports_df.to_csv(out_path, index=False)
print("Saved:", out_path, "|", len(reports_df), "rows")
reports_df.head(2)


### 3b. QA answer generation -> `results/qa_answers.csv`

Runs MedGemma on the QA test set **with** and **without** retrieved context (the
RAG ablation). Populate `context_lookup` with real retrieval output (CLIP or
ColPali) for final results; left empty here for a self-contained first pass.


In [ ]:
qa_csv = config.QA_DATA_DIR / "qa_test.csv"
if not qa_csv.exists():
    qa_csv = config.QA_DATA_DIR / "qa_dataset.csv"
assert qa_csv.exists(), "Missing QA dataset - run src/data/qa_builder.py locally first."
qa_df = pd.read_csv(qa_csv)

QA_LIMIT = 200  # cap for a quick run
qa_eval = qa_df.head(QA_LIMIT).copy()
print("QA eval rows:", len(qa_eval))

# study_id -> retrieved context string. Fill from your retrieval step for final runs.
context_lookup = {}

qa_records = []
for _, row in tqdm(qa_eval.iterrows(), total=len(qa_eval), desc="MedGemma QA"):
    try:
        img = Image.open(row["image_path"]).convert("RGB")
        ctx = context_lookup.get(row["study_id"])
        ans_no_ctx = medgemma_answer(img, row["question"], context=None)
        ans_with_ctx = medgemma_answer(img, row["question"], context=ctx) if ctx else ans_no_ctx
    except Exception as e:
        ans_no_ctx = ans_with_ctx = f"[ERROR] {e}"
    qa_records.append({
        "study_id": row["study_id"],
        "question": row["question"],
        "gold_answer": row.get("answer", ""),
        "answer_no_context": ans_no_ctx,
        "answer_with_context": ans_with_ctx,
    })

qa_out = pd.DataFrame(qa_records)
qa_out_path = config.RESULTS_DIR / "qa_answers.csv"
qa_out.to_csv(qa_out_path, index=False)
print("Saved:", qa_out_path)
qa_out.head(2)


### Free MedGemma memory before loading ColPali


In [ ]:
import gc
del medgemma
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 4. ColPali - build retrieval index

Uses `src/models/colpali_index.py`. Builds a multi-vector index over the training
corpus and saves it to `results/colpali_index/`.


## 3c. Build CLIP index (runs fast on Kaggle GPU)

In [ ]:
!pip install -q open_clip_torch faiss-cpu

from src.models.clip_index import CLIPRetriever

clip_retriever = CLIPRetriever(device="cuda")
clip_retriever.build_index(corpus_df, index_path=config.RESULTS_DIR / "clip_index")
print("CLIP index built.")

In [ ]:
from src.models.colpali_index import ColPaliRetriever

# corpus_df already loaded and remapped in cell 2b
print("Corpus:", len(corpus_df), "studies")

retriever = ColPaliRetriever(device="cuda")
retriever.build_index(corpus_df, index_path=config.RESULTS_DIR / "colpali_index")

### Test ColPali retrieval


In [ ]:
test_img = Image.open(eval_df.iloc[0]["image_path"]).convert("RGB")
study_ids, scores, reports = retriever.retrieve_by_image(test_img, k=3)
for i, (sid, sc, rep) in enumerate(zip(study_ids, scores, reports)):
    print(f"{i+1}. study={sid}  score={sc:.3f}")
    print("   ", rep[:200], "\n")


## 5. ColPali fine-tune (STRETCH GOAL)

LoRA fine-tune ColPali on chest X-ray (image, report) pairs so the retriever
learns CXR-specific alignment. Optional - only run once Phases 1-7 are solid.
Compare retrieval Recall@k before vs after.


In [ ]:
# --- ColPali LoRA fine-tune skeleton (STRETCH) ---
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset, DataLoader


class CXRPairDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        query = str(row["text"])[:300]  # report text acts as the query
        return img, query


def finetune_colpali(corpus_df, epochs=1, batch_size=4, lr=5e-5, lora_r=32):
    from colpali_engine.models import ColPali, ColPaliProcessor
    base = ColPali.from_pretrained("vidore/colpali-v1.3", torch_dtype=torch.bfloat16,
                                   device_map="cuda:0")
    processor = ColPaliProcessor.from_pretrained("vidore/colpali-v1.3")

    lora_cfg = LoraConfig(
        r=lora_r, lora_alpha=lora_r, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="FEATURE_EXTRACTION",
    )
    model = get_peft_model(base, lora_cfg)
    model.print_trainable_parameters()

    loader = DataLoader(CXRPairDataset(corpus_df), batch_size=batch_size,
                        shuffle=True, collate_fn=lambda b: b)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        for batch in tqdm(loader, desc=f"epoch {epoch+1}"):
            images = [b[0] for b in batch]
            queries = [b[1] for b in batch]
            img_in = processor.process_images(images).to(model.device)
            q_in = processor.process_queries(queries).to(model.device)
            img_emb = model(**img_in)
            q_emb = model(**q_in)
            # in-batch contrastive: diagonal of the score matrix is the positive pair
            scores = processor.score_multi_vector(q_emb, img_emb)
            labels = torch.arange(scores.size(0), device=scores.device)
            loss = torch.nn.functional.cross_entropy(scores, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        print(f"epoch {epoch+1} avg loss: {total/len(loader):.4f}")

    out = config.RESULTS_DIR / "colpali_finetuned"
    model.save_pretrained(out)
    print("Saved LoRA adapters to", out)
    return model


# Uncomment to run (slow):
# ft_model = finetune_colpali(corpus_df, epochs=1)


## 6. Download artifacts

Zip `results/` and download it, then unzip into your local repo's `results/` so
the Streamlit app and `run_eval.py` can use it.


In [ ]:
import shutil

zip_base = "/content/results_artifacts"
shutil.make_archive(zip_base, "zip", config.RESULTS_DIR)
print("Created:", zip_base + ".zip")

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("Download unavailable here:", e)
    print("Manually download:", zip_base + ".zip")


## Next steps

1. Unzip `results_artifacts.zip` into your local repo's `results/`.
2. Locally: `python src/models/clip_index.py` - build the CLIP index.
3. Locally: `python src/eval/run_eval.py` - compute comparison tables.
4. Locally: `streamlit run app/streamlit_app.py` - run the demo.
